In [ ]:
!pip install transformers datasets accelerate -q
import torch
import transformers
import datasets
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Torch version: 2.11.0+cu128
Transformers version: 5.13.1
GPU available: True
GPU name: Tesla T4


In [ ]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text
print("Total characters:", len(text))
print("\n--- Preview ---\n")
print(text[:500])

Total characters: 1115394

--- Preview ---

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
split_idx = int(len(text) * 0.9)
train_text = text[:split_idx]
val_text = text[split_idx:]
with open("train.txt", "w") as f:
    f.write(train_text)
with open("val.txt", "w") as f:
    f.write(val_text)
print("Train characters:", len(train_text))
print("Validation characters:", len(val_text))

Train characters: 1003854
Validation characters: 111540


In [ ]:
from transformers import GPT2Tokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
raw_datasets = load_dataset("text", data_files={"train": "train.txt", "validation": "val.txt"})
block_size = 128
def tokenize_function(examples):
    return tokenizer(examples["text"])
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // block_size) * block_size
    result = {
        k: [t[i:i+block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
train_dataset = lm_datasets["train"]
val_dataset = lm_datasets["validation"]
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Train blocks:", len(train_dataset))
print("Validation blocks:", len(val_dataset))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/35526 [00:00<?, ? examples/s]

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]

Map:   0%|          | 0/35526 [00:00<?, ? examples/s]

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]

Train blocks: 2061
Validation blocks: 244


In [ ]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.to("cuda")
training_args = TrainingArguments(
    output_dir="./gpt2-shakespeare",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)
print("Model loaded and Trainer is set up. Ready to train.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded and Trainer is set up. Ready to train.


In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.147489,3.954704
2,3.908595,3.920806
3,3.820061,3.910491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=774, training_loss=4.010856450990189, metrics={'train_runtime': 404.8556, 'train_samples_per_second': 15.272, 'train_steps_per_second': 1.912, 'total_flos': 403892158464000.0, 'train_loss': 4.010856450990189, 'epoch': 3.0})

In [ ]:
model.eval()
prompt = "To be, or not to be"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    input_ids,
    max_length=100,
    num_return_sequences=1,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id
)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

To be, or not to be: if that is so.LEONTESHER :So this I thought thou didst wrong me; now here's my good lady--A look with it and a kiss of love in the midstOf all thy dreams!JULIET:'The truth am't but for what you have told him,'Why were he such merry eyes as these?And why was she poor when ever an eyeThou art no more than child againOr beggar unto death
